# Day 3: ML Pipeline and Bias–Variance
## Student Practice Notebook

**Dataset:** TabReD Delivery ETA  
**Task:** Regression  
**Evaluation metric:** RMSE  
**Environment:** Python 3.13.x, Jupyter Notebook, `.venv`

### Scenario
A logistics organisation wants to estimate delivery-related outcomes using historical data. Your task is to prepare a basic machine-learning workflow, use the dataset's supplied partitions, train a simple baseline and evaluate how it performs on data it did not learn from.

### Learning objectives
By the end of this notebook, you should be able to:
- Load feature arrays, target values and official split indices.
- Explain the roles of training, validation and test partitions.
- Build a basic scikit-learn `Pipeline`.
- Calculate RMSE and compare training with validation performance.
- Identify possible data leakage risks and explain why generalisation matters.

> **Important:** The dataset metadata confirms a regression task and RMSE scoring. It does not establish the target's business unit or explain all feature encodings. Do not label RMSE as minutes or hours unless you have verified documentation.


## 1. Environment setup

Run this notebook from the project root, where the `data` folder is located.

If required packages are missing, activate your project `.venv` and install the course dependencies using the project's `requirements-py313.txt`.

The notebook uses:
- NumPy
- pandas
- scikit-learn



In [ ]:
# Task 1: Import the libraries you need.

# TODO:
# Import Path from pathlib
# Import json
# Import numpy as np
# Import pandas as pd
# Import the required scikit-learn classes:
#   SimpleImputer, DummyRegressor, Pipeline, StandardScaler
# Import mean_squared_error from sklearn.metrics

# Write your imports below:


## 2. Set the dataset location and inspect the metadata

The expected dataset location is:

`data/TabReD_Delivery_ETA/delivery-eta`

The `info.json` file confirms:
- Task type: regression
- Scoring metric: RMSE

### Your task
1. Create a `Path` object for the dataset directory.
2. Open `info.json`.
3. Load the JSON content.
4. Print it in a readable format.
5. Confirm that the task is regression and the metric is RMSE.


In [ ]:
# Task 2: Define the dataset path and load info.json.

# Hint:
# DATA_DIR = Path("data/TabReD_Delivery_ETA/delivery-eta")
# Use open(..., encoding="utf-8") and json.load(...)

# TODO: Write your code below.


## 3. Load and inspect the arrays

The dataset contains the following arrays:

| File | Expected shape | Data type | Description supported by current metadata |
|---|---:|---|---|
| `x_num.npy` | (416451, 221) | float32 | Numerical features |
| `x_cat.npy` | (416451, 1) | int64 | Integer-coded categorical feature |
| `x_bin.npy` | (416451, 1) | float32 | Binary feature |
| `x_meta.npy` | (416451, 2) | int64 | Metadata; meaning not established |
| `y.npy` | (416451,) | float32 | Regression target |

### Your task
Load each array with `np.load`. Use `mmap_mode="r"` and `allow_pickle=False` where appropriate. Print the shape and data type of every array.

**Do not assume** that integer-coded categories are continuous measurements. Do not use `x_meta` as a model feature until its meaning is documented.


In [ ]:
# Task 3: Load the five NumPy arrays and inspect them.

# Suggested pattern:
# x_num = np.load(DATA_DIR / "x_num.npy", mmap_mode="r", allow_pickle=False)
#
# Repeat for x_cat, x_bin, x_meta and y.
#
# TODO: Write your code below.


## 4. Inspect sample records and target values

Before modelling, inspect a few rows and check for missing or non-finite values.

### Your task
1. Display the first five rows of each feature array and the first five target values.
2. Count missing values in the target.
3. Use `pd.Series(y).describe()` to inspect the target distribution.
4. Record what you can conclude—and what you cannot conclude—from these summaries.

Do not infer the target's real-world unit from its numerical magnitude alone.


In [ ]:
# Task 4: Inspect sample values and the target summary.

# TODO:
# - Print the first five rows/values of each array.
# - Count NaN values in y.
# - Display descriptive statistics for y.



## 5. Load the official `default` split indices

Use the supplied files in:

`data/TabReD_Delivery_ETA/delivery-eta/splits/default/`

Expected partition sizes:

| Partition | Records |
|---|---:|
| Training | 279,415 |
| Validation | 34,174 |
| Testing | 36,927 |

These files contain row indices. They are not copies of the feature data.

### Your task
1. Load `train.npy`, `val.npy` and `test.npy`.
2. Print each partition's record count, minimum index and maximum index.
3. Confirm that the indices are within the available row range.
4. Check that there are no duplicate indices within a partition.
5. Check that the three partitions do not overlap.

**Important observation:** the maximum supplied index is 350,515, although the arrays contain 416,451 rows. Preserve the supplied partitions exactly. Do not add the unreferenced rows to any partition.


In [ ]:
# Task 5: Load the official split indices.

# Hint:
# SPLIT_DIR = DATA_DIR / "splits" / "default"
# train_idx = np.load(SPLIT_DIR / "train.npy", allow_pickle=False)
#
# TODO: Load all three index arrays and inspect their sizes/ranges.


In [ ]:
# Task 6: Validate the split indices.

# TODO:
# 1. Check that all indices are >= 0 and < len(y).
# 2. Check for duplicate indices in each split using np.unique.
# 3. Check that train, validation and test indices do not overlap.
# 4. Calculate how many dataset rows are not referenced by these splits.
#
# Hint:
# set(train_idx.tolist()).isdisjoint(set(val_idx.tolist()))


## 6. Prepare the baseline feature matrix

For this introductory exercise, use:
- `x_num`: 221 numerical columns
- `x_bin`: 1 binary column

Combine these into one matrix with 222 columns.

For this exercise, leave `x_cat` and `x_meta` out of the model:
- The category mapping for `x_cat` has not been established in the supplied metadata.
- The meaning of the two `x_meta` columns has not been established.

### Your task
1. Convert the selected feature arrays to `float32`.
2. Combine them column-wise using `np.concatenate(..., axis=1)`.
3. Convert the target to a NumPy array.
4. Use the official indices to create `X_train`, `X_val`, `X_test`, `y_train`, `y_val` and `y_test`.
5. Print the resulting shapes.


In [ ]:
# Task 7: Create the baseline feature matrix and target vector.

# TODO:
# X_all should contain x_num followed by x_bin.
# y_all should contain the target values.
#
# Hint:
# X_all = np.concatenate(
#     [np.asarray(x_num, dtype=np.float32),
#      np.asarray(x_bin, dtype=np.float32)],
#     axis=1
# )
#
# TODO: Write your code below.


In [ ]:
# Task 8: Select the three partitions using the supplied indices.

# TODO:
# X_train = X_all[train_idx]
# y_train = y_all[train_idx]
#
# Repeat for validation and test.
# Print the shapes of all six arrays.


## 7. Build a scikit-learn Pipeline

A pipeline joins preprocessing and a model into one workflow.

For this exercise, use the following sequence:

1. `SimpleImputer(strategy="median")`
2. `StandardScaler()`
3. `DummyRegressor(strategy="mean")`

The imputer and scaler must be fitted using the training partition only. When you call `pipeline.fit(X_train, y_train)`, scikit-learn learns the preprocessing parameters from the training data and then fits the baseline estimator.

`DummyRegressor` is a simple benchmark. It predicts the mean target value learned from the training data for every input row. It is not intended to be a production-quality delivery model.

### Your task
Create the pipeline, fit it on the training data and print the pipeline structure.


In [ ]:
# Task 9: Create and fit the pipeline.

# TODO:
# Create a Pipeline with these named steps:
#   ("imputer", SimpleImputer(strategy="median"))
#   ("scaler", StandardScaler())
#   ("model", DummyRegressor(strategy="mean"))
#
# Fit it using X_train and y_train.
# Print the fitted pipeline.


## 8. Calculate RMSE

RMSE is the square root of the mean squared prediction error:

\[
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}
\]

Use `mean_squared_error` and `np.sqrt` to calculate RMSE. This approach works across a wider range of scikit-learn versions than relying on a newer dedicated RMSE helper.

### Your task
1. Write a function named `calculate_rmse`.
2. Generate predictions for the training and validation partitions.
3. Calculate both RMSE values.
4. Display a small results table with partition name, record count and RMSE.

Report the score in target units. Do not call it minutes or hours unless the target unit has been verified.


In [ ]:
# Task 10: Create an RMSE function.

def calculate_rmse(y_true, y_pred):
    # TODO: Return the square root of mean_squared_error.
    pass


In [ ]:
# Task 11: Generate training and validation predictions.

# TODO:
# train_predictions = pipeline.predict(X_train)
# val_predictions = pipeline.predict(X_val)
#
# Calculate train_rmse and val_rmse using calculate_rmse().
# Create and display a pandas DataFrame containing:
# Partition, Records, RMSE.


## 9. Interpret the results

Answer these questions in your own words:

1. Why is the training RMSE not enough to judge a model?
2. What does the validation RMSE tell you?
3. If training RMSE is much lower than validation RMSE, what might that indicate?
4. If both values are high, what might that indicate?
5. Why does a similar training and validation score not automatically mean the model is useful?

Remember: this is a `DummyRegressor` baseline. It does not learn relationships between the input features and target, so it cannot demonstrate every overfitting or underfitting pattern.


In [ ]:
# Task 12: Write your interpretation.

# Replace the placeholder text with your own observations.
training_vs_validation_interpretation = "TODO: Explain what your two RMSE values suggest."
print(training_vs_validation_interpretation)


## 10. Final test evaluation

The test partition should remain untouched during model selection. In a real workflow, use the training and validation partitions to develop and compare candidate models. Evaluate the final selected workflow on the test partition once the choices are complete.

For this guided exercise, evaluate the baseline on the test partition to practise the calculation. Do not use the test result to tune or select a model.


In [ ]:
# Task 13: Evaluate the fitted baseline on the test partition.

# TODO:
# test_predictions = pipeline.predict(X_test)
# test_rmse = calculate_rmse(y_test, test_predictions)
# Print the test RMSE.


## 11. Data leakage: identify the risk

For each example, decide whether it creates a leakage risk and explain why.

1. A scaler is fitted on the complete dataset before the train/validation/test split.
2. A delivery completion timestamp is used to predict an outcome that must be estimated before delivery.
3. A pipeline is fitted on training data and then used to transform validation data.
4. A team repeatedly changes its model based on test-set scores.

Use the Markdown cell below to record your answers.


In [ ]:
# Task 14: Record your data leakage answers.

leakage_answers = {
    "Example 1": "TODO",
    "Example 2": "TODO",
    "Example 3": "TODO",
    "Example 4": "TODO",
}

for example, answer in leakage_answers.items():
    print(f"{example}: {answer}")


## 12. Business interpretation: delivery ETA reliability

A logistics organisation wants to use model predictions to support planning and customer communication.

Write a short business interpretation covering:
- Why unseen-data evaluation matters.
- What could go wrong if a misleadingly optimistic model score is trusted.
- Why the target definition and units must be understood before explaining RMSE to business stakeholders.
- What additional information you would request before approving the model for operational use.


In [ ]:
# Task 15: Write your business recommendation.

business_recommendation = """
TODO:
Write a short recommendation to the logistics operations manager.
Explain what the current baseline does and does not establish.
Do not claim that it is ready for production.
"""

print(business_recommendation)


## 13. Knowledge check

Answer these questions without looking back at the notes:

1. What is the difference between training, validation and test data?
2. What problem does a scikit-learn `Pipeline` help reduce?
3. Why is `DummyRegressor` useful even though it is not a strong predictive model?
4. What is overfitting?
5. What is underfitting?
6. What is data leakage?
7. Why should the test partition not be repeatedly used for model selection?

### Submission checklist
- [ ] All five arrays loaded and inspected.
- [ ] Official split indices loaded and validated.
- [ ] Baseline feature matrix created using `x_num` and `x_bin`.
- [ ] Training, validation and test arrays prepared.
- [ ] Pipeline created and fitted on training data only.
- [ ] Training and validation RMSE calculated.
- [ ] Test RMSE calculated as a final practice evaluation.
- [ ] Data leakage examples answered.
- [ ] Business interpretation completed.

### Dataset and modelling cautions
- Use the supplied `default` split without changing its membership.
- Do not assign the unreferenced rows to a partition.
- Do not include `x_cat` or `x_meta` in this introductory baseline without verified encoding/meaning.
- Do not infer target units or transformation from the numerical values alone.
- The baseline is for learning workflow structure, not production deployment.
